# RNN Models
Composed of 3 models (one trained on all data, one on critics, one on users)

## Setup
Ensure that pytorch is installed first, as well as any other dependencies

In [2]:
%pip install langdetect kaggle

import copy
import io
import random
import re
import shutil
import time
from time import perf_counter
import unicodedata
import urllib.request
import zipfile
from pathlib import Path
import langdetect
from tempfile import TemporaryDirectory
import subprocess
from sysconfig import get_path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Subset, random_split, DataLoader, Dataset, ConcatDataset
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Selected device:", DEVICE)

PAD, UNK = 0, 1
MAX_SENTENCE_TOKENS = 9
MAX_PAIRS = 3000
BATCH_SIZE = 64
EPOCHS=50

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Selected device: cpu


### Helper Functions

In [ ]:
@torch.no_grad
def predict_score(review, model, vocab, device=DEVICE):
    model.eval()
    tokens = vocab.encode(review)
    src = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)
    src_lengths = torch.tensor([len(src)], dtype=torch.long).to(device)
    predictions = model(src, src_lengths)
    score = predictions.squeeze(-1).item()
    return score

## Vocabulary and Minibatches

### Vocab

In [4]:
class Vocab:
    def __init__(self):
        self.word2id = {"<pad>": PAD, "<unk>": UNK}
        self.id2word = {PAD: "<pad>", UNK: "<unk>"}
    
    def __len__(self):
        return len(self.word2id)
    
    def add_sentence(self, sentence):
        for word in sentence.split():
            if word not in self.word2id:
                id = len(self.word2id)
                self.word2id[word] = id
                self.id2word[id] = word

    def encode(self, sentence):
        return [self.word2id.get(word, UNK) for word in sentence.split()]

### Dataset

In [5]:
class MovieReviewDataset(Dataset):
    def __init__(self, reviews, scores, vocab):
        self.reviews = reviews
        self.scores = scores
        self.vocab = vocab
    
    def __len__(self):
        return len(self.reviews)
    
    def __getitem__(self, index):
        text = self.vocab.encode(self.reviews[index])
        label = float(self.scores[index])
        return text, label

### Collate

In [30]:
def collate(batch):
    src_sequences = []
    src_lengths = []
    targets = []
    for src, target in batch:
        src_sequences.append(torch.tensor(src, dtype=torch.long))
        src_lengths.append(len(src))
        targets.append(target)
    src = pad_sequence(src_sequences, batch_first=True, padding_value=PAD)
    src_lengths = torch.tensor(src_lengths, dtype=torch.long)
    targets = torch.tensor(targets, dtype=torch.float32)
    return src, src_lengths, targets

## Preprocessing

In [ ]:
# Grab reviews of specific type
def reviews_by_type(df, review_type_column="review_type", target_type="BOTH"):
    if review_type_column not in df.columns:
        raise ValueError(
            f"Review type column {review_type_column!r} not found. "
            f"Available columns: {', '.join(map(str, df.columns))}"
        )

    types = df[review_type_column].astype(str).str.strip().str.casefold()
    return df.loc[types == target_type.strip().casefold()].copy()

# Grab COUNT samples that are in english
def english_sample(group, review_column, count):
    # Randomize the order so we don't always inspect the same rows first.
    shuffled = group.sample(frac=1, random_state=SEED)
    english_indices = []

    for index, text in shuffled[review_column].items():
        if isinstance(text, str) and text.strip():
            try:
                if langdetect.detect(text) == "en":
                    english_indices.append(index)
                    if len(english_indices) == count:
                        break
            except langdetect.LangDetectException:
                pass

    if len(english_indices) < count:
        raise ValueError(
            f"Found only {len(english_indices)} English reviews; "
            f"needed {count}. Increase the available data or lower the requested count."
        )

    return shuffled.loc[english_indices].copy()

# Creates the datasets for training, validation, and testing
def make_loaders(dataset, batch_size=BATCH_SIZE, total_entries=None):
    if total_entries is None:
        total_entries = len(dataset)

    if total_entries < 10:
        raise ValueError("total_entries must be at least 10.")
    if total_entries > len(dataset):
        raise ValueError(
            f"Requested {total_entries} entries, but the dataset has "
            f"only {len(dataset)}."
        )

    selection_generator = torch.Generator().manual_seed(SEED)
    indices = torch.randperm(
        len(dataset), generator=selection_generator
    )[:total_entries].tolist()

    selected_dataset = Subset(dataset, indices)

    train_size = int(0.8 * total_entries)
    val_size = int(0.1 * total_entries)
    test_size = total_entries - train_size - val_size

    train_set, val_set, test_set = random_split(
        selected_dataset,
        [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(SEED + 1),
    )

    return (
        DataLoader(
            train_set, batch_size=batch_size, shuffle=True, collate_fn=collate
        ),
        DataLoader(
            val_set, batch_size=batch_size, collate_fn=collate
        ),
        DataLoader(
            test_set, batch_size=batch_size, collate_fn=collate
        ),
    )

def prepare_group(
    df,
    type_value,
    entries_per_group,
    review_type_column="review_type",
    review_column="quote",
    score_column="score",
    score_multiplier=1,
):
    group = reviews_by_type(
        df,
        review_type_column=review_type_column,
        target_type=type_value,
    )

    group[score_column] = pd.to_numeric(group[score_column], errors="coerce")
    group = group.dropna(subset=[review_column, score_column]).copy()
    group[score_column] *= score_multiplier
    group = group[group[score_column].between(0, 10)]

    # Select the requested number of English reviews from this group.
    group = english_sample(group, review_column, entries_per_group)

    reviews = group[review_column].astype(str).tolist()
    scores = group[score_column].tolist()
    return reviews, scores

# Builds loaders for groups (users, critics)
def build_group_loaders(
    df,
    type_value,
    entries_per_group,
    review_type_column="review_type",
    review_column="quote",
    score_column="score",
    score_multiplier=1,
):
    reviews, scores = prepare_group(
        df,
        type_value=type_value,
        entries_per_group=entries_per_group,
        review_type_column=review_type_column,
        review_column=review_column,
        score_column=score_column,
        score_multiplier=score_multiplier,
    )

    vocab = Vocab()
    for review in reviews:
        vocab.add_sentence(review)

    dataset = MovieReviewDataset(reviews, scores, vocab)
    loaders = make_loaders(dataset, total_entries=entries_per_group)
    return loaders, vocab

# Build loader that uses user/critic split
def build_both_loaders(
    df,
    entries_per_source,
    review_type_column="review_type",
    review_column="quote",
    score_column="score",
    user_score_multiplier=1/10,
    critic_score_multiplier=1/10,
    batch_size=BATCH_SIZE,
):
    user_reviews, user_scores = prepare_group(
        df,
        type_value="user",
        entries_per_group=entries_per_source,
        review_type_column=review_type_column,
        review_column=review_column,
        score_column=score_column,
        score_multiplier=user_score_multiplier,
    )

    critic_reviews, critic_scores = prepare_group(
        df,
        type_value="critic",
        entries_per_group=entries_per_source,
        review_type_column=review_type_column,
        review_column=review_column,
        score_column=score_column,
        score_multiplier=critic_score_multiplier,
    )

    # Both sources share one vocabulary for the combined model.
    both_vocab = Vocab()
    for review in user_reviews + critic_reviews:
        both_vocab.add_sentence(review)

    user_dataset = MovieReviewDataset(user_reviews, user_scores, both_vocab)
    critic_dataset = MovieReviewDataset(critic_reviews, critic_scores, both_vocab)

    # Split each source separately so every split has equal user/critic counts.
    user_splits = make_loaders(user_dataset, total_entries=entries_per_source)
    critic_splits = make_loaders(critic_dataset, total_entries=entries_per_source)

    combined_sets = [
        ConcatDataset([user_splits[i].dataset, critic_splits[i].dataset])
        for i in range(3)
    ]

    both_loaders = (
        DataLoader(
            combined_sets[0],
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate,
        ),
        DataLoader(
            combined_sets[1],
            batch_size=batch_size,
            collate_fn=collate,
        ),
        DataLoader(
            combined_sets[2],
            batch_size=batch_size,
            collate_fn=collate,
        ),
    )

    return both_loaders, both_vocab

# Load training, validation, testing sets

In [ ]:
kaggle_exe = shutil.which("kaggle")
if kaggle_exe is None:
    raise FileNotFoundError("Kaggle CLI not found. Run %pip install kaggle.")

dataset_id = "davutb/metacritic-movies"
csv_filename = "movies_reviews.csv"

# For "both" group, this means 20,000, because there's 10,000 per GROUP
entries_per_group = 10_000

review_column = "quote"
score_column = "score"
review_type_column = "review_type"

# Load kaggle dataset temporarily
with TemporaryDirectory() as temp_dir:
    subprocess.run(
        [
            kaggle_exe,
            "datasets", "download", dataset_id,
            "-f", csv_filename,
            "-p", temp_dir,
            "--unzip",
        ],
        check=True,
    )

    csv_files = list(Path(temp_dir).rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV found; check {csv_filename!r}.")

    # Read the CSV once
    df = pd.read_csv(csv_files[0])
    print(df[review_type_column].value_counts(dropna=False))

    user_loaders, user_vocab = build_group_loaders(
        df,
        type_value="user",
        entries_per_group=entries_per_group,
        review_type_column=review_type_column,
        review_column=review_column,
        score_column=score_column,
        score_multiplier=1/10,
    )

    critic_loaders, critic_vocab = build_group_loaders(
        df,
        type_value="critic",
        entries_per_group=entries_per_group,
        review_type_column=review_type_column,
        review_column=review_column,
        score_column=score_column,
        score_multiplier=1/10,
    )

    both_loaders, both_vocab = build_both_loaders(
        df,
        entries_per_source=entries_per_group,
        review_type_column=review_type_column,
        review_column=review_column,
        score_column=score_column,
        user_score_multiplier=1/10,
        critic_score_multiplier=1/10,
    )

# Get loaders for user, critic, and both groups
user_train_loader, user_val_loader, user_test_loader = user_loaders
critic_train_loader, critic_val_loader, critic_test_loader = critic_loaders
both_train_loader, both_val_loader, both_test_loader = both_loaders

category_loaders = {
    "User": user_loaders,
    "Critic": critic_loaders,
    "Both": both_loaders,
}

FileNotFoundError: Kaggle CLI not found. Run %pip install kaggle.

## RNN Model

### Cell

In [10]:
class RNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_linear = nn.Linear(input_size, hidden_size)
        self.hidden_linear = nn.Linear(hidden_size, hidden_size)

    def forward(self, x_t, h_prev):
        input = self.input_linear(x_t)
        hidden = self.hidden_linear(h_prev)
        state = input + hidden
        return torch.tanh(state)

### Encoder

In [11]:
class Encoder(nn.Module):
    def __init__(self, source_vocab_size, embedding_dim,
                 hidden_size, dropout=0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(
            source_vocab_size, embedding_dim, padding_idx=PAD
        )
        self.dropout = nn.Dropout(dropout)
        self.cell = RNNCell(input_size=embedding_dim, hidden_size=hidden_size)

    def forward(self, src, src_length):
        embedded = self.dropout(self.embedding(src))
        batch_size = src.size(0)
        h = embedded.new_zeros(batch_size, self.hidden_size)
        for t in range(src.size(1)):
            proposal = self.cell(embedded[:, t], h)
            mask = (t < src_length).unsqueeze(1)
            h = torch.where(mask, proposal, h)
        return h

### Score Regressor

In [31]:
class ScoreRegressor(nn.Module):
    def __init__(self, source_vocab_size, embedding_dim,
                 hidden_size, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(source_vocab_size, embedding_dim, hidden_size, dropout)
        self.regressor = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, src, src_lengths):
        encoder = self.encoder(src, src_lengths)
        predictions = self.regressor(encoder)
        return predictions

## Training and Validation

### Training One Epoch

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, clip_value=1.0):
    model.train()
    total_loss = 0.0
    total_mae = 0.0
    total_samples = 0

    for src, src_lengths, labels in loader:
        src = src.to(device)
        src_lengths = src_lengths.to(device)
        labels = labels.to(device).float()

        optimizer.zero_grad()
        predictions = model(src, src_lengths).squeeze(-1)
        loss = criterion(predictions, labels)
        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=clip_value)
        optimizer.step()

        batch_size = src.size(0)
        total_loss += loss.item() * batch_size
        total_mae += torch.abs(predictions - labels).sum().item()
        total_samples += batch_size

    return total_loss / total_samples, total_mae / total_samples

### Evaluate

In [27]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_mae = 0.0
    total_samples = 0

    for src, src_lengths, labels in loader:
        src = src.to(device)
        src_lengths = src_lengths.to(device)
        labels = labels.to(device).float()

        predictions = model(src, src_lengths).squeeze(-1)
        loss = criterion(predictions, labels)

        batch_size = src.size(0)
        total_loss += loss.item() * batch_size
        total_mae += torch.abs(predictions - labels).sum().item()
        total_samples += batch_size

    return total_loss / total_samples, total_mae / total_samples

### Fit Model

In [ ]:
def fit_model(model, train_loader, val_loader, epochs=EPOCHS, patience=4, lr=1e-3, 
              weight_decay=1e-4, clip_value=1.0, device=DEVICE, verbose=True):
    model.to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )
    criterion = nn.MSELoss()

    history = {
        "train_loss": [],
        "train_mae": [],
        "val_loss": [],
        "val_mae": [],
    }

    best_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_mae = train_one_epoch(
            model, train_loader, criterion, optimizer, device, clip_value
        )
        va_loss, va_mae = evaluate(
            model, val_loader, criterion, device
        )

        for key, value in (
            ("train_loss", tr_loss),
            ("train_mae", tr_mae),
            ("val_loss", va_loss),
            ("val_mae", va_mae),
        ):
            history[key].append(value)

        if va_loss < best_loss:
            best_loss = va_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if verbose:
            print(
                f"Epoch {epoch:02d}/{epochs} | "
                f"train loss {tr_loss:.4f} | val loss {va_loss:.4f} | "
                f"train MAE {tr_mae:.2f} | val MAE {va_mae:.2f}"
            )

        # Early stopping with patience
        if epochs_without_improvement >= patience:
            if verbose:
                print(f"Early stopping after epoch {epoch}.")
            break

    model.load_state_dict(best_state)
    return history

## Executables

### RNN (CRITICS + USERS)

In [ ]:
models = {}
histories = {}

# Train three independent models.
models["User"] = ScoreRegressor(
    source_vocab_size=len(user_vocab),
    embedding_dim=64,
    hidden_size=64,
)

print("\n" + "=" * 15 + " USER MODEL " + "=" * 15)

histories["User"] = fit_model(
    models["User"],
    user_train_loader,
    user_val_loader,
    epochs=EPOCHS,
)

models["Critic"] = ScoreRegressor(
    source_vocab_size=len(critic_vocab),
    embedding_dim=64,
    hidden_size=64,
)

print("\n" + "=" * 15 + " CRITIC MODEL " + "=" * 15)

histories["Critic"] = fit_model(
    models["Critic"],
    critic_train_loader,
    critic_val_loader,
    epochs=EPOCHS,
)

models["Both"] = ScoreRegressor(
    source_vocab_size=len(both_vocab),
    embedding_dim=64,
    hidden_size=64,
)

print("\n" + "=" * 15 + " BOTH MODEL " + "=" * 15)

histories["Both"] = fit_model(
    models["Both"],
    both_train_loader,
    both_val_loader,
    epochs=EPOCHS,
)

# Evaluate each model after fit_model has restored its best validation checkpoint.
criterion = nn.MSELoss()
test_loaders = {
    "User": user_test_loader,
    "Critic": critic_test_loader,
    "Both": both_test_loader,
}

best_results = {}

for name, model in models.items():
    history = histories[name]
    best_index = min(
        range(len(history["val_loss"])),
        key=lambda i: history["val_loss"][i],
    )

    test_loss, test_mae = evaluate(
        model,
        test_loaders[name],
        criterion,
        DEVICE,
    )

    best_results[name] = {
        "best_epoch": best_index + 1,
        "val_loss": history["val_loss"][best_index],
        "val_mae": history["val_mae"][best_index],
        "test_loss": test_loss,
        "test_mae": test_mae,
    }

    print(
        f"{name}: best epoch {best_index + 1} | "
        f"val loss {history['val_loss'][best_index]:.4f} | "
        f"val mae {history['val_mae'][best_index]:.2f} | "
        f"test loss {test_loss:.4f} | "
        f"test mae {test_mae:.2f}"
    )

NameError: name 'user_vocab' is not defined